<a href="https://colab.research.google.com/github/cara-jvr/mit-805-group-project/blob/main/notebooks/Group12_01_Data_exploration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**MIT 805 Big Data Project** <br>
Group 12 <br>
Unarine <br>
Cara Janse van Rensburg (18002189)

**1. Start-Up:** Creating Spark Session and importing required libraries

In [1]:
from pyspark.sql import SparkSession

# creating a Spark Session Object
spark = (
    SparkSession.builder
    .appName("Big-Data-Project")
    .master("local[*]")
    .config("spark.driver.memory", "4g")
    .getOrCreate()
)
sc = spark.sparkContext
spark

In [37]:
# import libraries
from pyspark.sql.functions import split, explode, col, desc, sum as _sum, when, count, to_timestamp, lit, datediff, timestamp_diff, unix_timestamp
from pyspark.sql import functions as F
import pyarrow.parquet as pq
import glob
import pandas as pd
import os
from google.colab import drive
import seaborn as sns
import matplotlib.pyplot as plt

**2. Loading the data**

In [3]:
# reading files from Google Drive
drive.mount('/content/drive')
file_path = "drive/MyDrive/Big_data_project/data"
os.listdir(file_path)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


['fhvhv_tripdata_2023-01.parquet',
 'fhvhv_tripdata_2023-02.parquet',
 'fhvhv_tripdata_2023-03.parquet',
 'fhvhv_tripdata_2023-04.parquet',
 'fhvhv_tripdata_2023-05.parquet',
 'fhvhv_tripdata_2023-06.parquet',
 'fhvhv_tripdata_2023-08.parquet',
 'fhvhv_tripdata_2023-09.parquet',
 'fhvhv_tripdata_2023-11.parquet',
 'fhvhv_tripdata_2023-12.parquet',
 'fhvhv_tripdata_2023-07.parquet',
 'fhvhv_tripdata_2023-10.parquet']

In [4]:
df = spark.read.parquet(file_path)

**3. Dataset Characteristics**

In [5]:
# Getting the column names and data types
df.printSchema()

root
 |-- hvfhs_license_num: string (nullable = true)
 |-- dispatching_base_num: string (nullable = true)
 |-- originating_base_num: string (nullable = true)
 |-- request_datetime: timestamp_ntz (nullable = true)
 |-- on_scene_datetime: timestamp_ntz (nullable = true)
 |-- pickup_datetime: timestamp_ntz (nullable = true)
 |-- dropoff_datetime: timestamp_ntz (nullable = true)
 |-- PULocationID: long (nullable = true)
 |-- DOLocationID: long (nullable = true)
 |-- trip_miles: double (nullable = true)
 |-- trip_time: long (nullable = true)
 |-- base_passenger_fare: double (nullable = true)
 |-- tolls: double (nullable = true)
 |-- bcf: double (nullable = true)
 |-- sales_tax: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: double (nullable = true)
 |-- tips: double (nullable = true)
 |-- driver_pay: double (nullable = true)
 |-- shared_request_flag: string (nullable = true)
 |-- shared_match_flag: string (nullable = true)
 |-- access_a_ride_f

In [6]:
# number of rows
rows = df.count()
print(f"Number of rows: {rows/10**6} million")

Number of rows: 232.49002 million


In [7]:
# size of compressed raw file
compressed_gb = 0

for root, dirs, files in os.walk(file_path):
    for file in files:
        if file.endswith(".parquet"):
            compressed_gb += os.path.getsize(os.path.join(root, file))

total_gb = compressed_gb / (1024**3)

print(f"Total dataset size: {total_gb:.2f} GB")

Total dataset size: 5.42 GB


In [8]:
# size of uncompressed raw file

def get_parquet_uncompressed_size(file_path):
    """Extracts the total uncompressed size from a Parquet file footer metadata."""
    try:
        # Read only the metadata footer, avoiding loading the actual data into memory
        metadata = pq.read_metadata(file_path)

        # Sum up the uncompressed size of all row groups
        total_bytes = sum(metadata.row_group(i).total_byte_size for i in range(metadata.num_row_groups))
        return total_bytes
    except Exception as e:
        print(f"Error reading metadata for {file_path}: {e}")
        return 0

total_uncompressed_bytes = 0
file_count = 0

# Recursively walk through the directory tree
for root, dirs, files in os.walk(file_path):
    for file in files:
        if file.endswith('.parquet') or file.endswith('.parq'):
            full_path = os.path.join(root, file)

            # Get the uncompressed size in bytes
            uncompressed_bytes = get_parquet_uncompressed_size(full_path)
            total_uncompressed_bytes += uncompressed_bytes
            file_count += 1

            # Convert bytes to Gigabytes (GB)
            size_in_gb = uncompressed_bytes / (1024 ** 3)

            # Print individual file metrics (truncated names for alignment)
            display_name = file if len(file) <= 47 else f"...{file[-44:]}"
            print(f"{display_name:<50} | {size_in_gb:>21.4f} GB")

# Final summary conversion
total_uncompressed_gb = total_uncompressed_bytes / (1024 ** 3)

print("=" * 75)
print(f"Total Parquet Files Found: {file_count}")
print(f"Total Uncompressed Size:   {total_uncompressed_gb:.4f} GB")


fhvhv_tripdata_2023-01.parquet                     |                0.7929 GB
fhvhv_tripdata_2023-02.parquet                     |                0.7651 GB
fhvhv_tripdata_2023-03.parquet                     |                0.8706 GB
fhvhv_tripdata_2023-04.parquet                     |                0.8126 GB
fhvhv_tripdata_2023-05.parquet                     |                0.8499 GB
fhvhv_tripdata_2023-06.parquet                     |                0.8261 GB
fhvhv_tripdata_2023-08.parquet                     |                0.5358 GB
fhvhv_tripdata_2023-09.parquet                     |                0.5454 GB
fhvhv_tripdata_2023-11.parquet                     |                0.5366 GB
fhvhv_tripdata_2023-12.parquet                     |                0.5570 GB
fhvhv_tripdata_2023-07.parquet                     |                0.8181 GB
fhvhv_tripdata_2023-10.parquet                     |                0.5614 GB
Total Parquet Files Found: 12
Total Uncompressed Size:   8.4715 

In [9]:
df.limit(1).show()

+-----------------+--------------------+--------------------+-------------------+-------------------+-------------------+-------------------+------------+------------+----------+---------+-------------------+-----+----+---------+--------------------+-----------+----+----------+-------------------+-----------------+------------------+----------------+--------------+
|hvfhs_license_num|dispatching_base_num|originating_base_num|   request_datetime|  on_scene_datetime|    pickup_datetime|   dropoff_datetime|PULocationID|DOLocationID|trip_miles|trip_time|base_passenger_fare|tolls| bcf|sales_tax|congestion_surcharge|airport_fee|tips|driver_pay|shared_request_flag|shared_match_flag|access_a_ride_flag|wav_request_flag|wav_match_flag|
+-----------------+--------------------+--------------------+-------------------+-------------------+-------------------+-------------------+------------+------------+----------+---------+-------------------+-----+----+---------+--------------------+-----------+--

**4. Data Quality Analysis & Data Cleaning**

In [10]:
#Missing Values
missing = df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df.columns
])

missing.show()

+-----------------+--------------------+--------------------+----------------+-----------------+---------------+----------------+------------+------------+----------+---------+-------------------+-----+---+---------+--------------------+-----------+----+----------+-------------------+-----------------+------------------+----------------+--------------+
|hvfhs_license_num|dispatching_base_num|originating_base_num|request_datetime|on_scene_datetime|pickup_datetime|dropoff_datetime|PULocationID|DOLocationID|trip_miles|trip_time|base_passenger_fare|tolls|bcf|sales_tax|congestion_surcharge|airport_fee|tips|driver_pay|shared_request_flag|shared_match_flag|access_a_ride_flag|wav_request_flag|wav_match_flag|
+-----------------+--------------------+--------------------+----------------+-----------------+---------------+----------------+------------+------------+----------+---------+-------------------+-----+---+---------+--------------------+-----------+----+----------+-------------------+-----

In [16]:
# checking for duplicates
duplicates_df = (
    df.groupBy("hvfhs_license_num", "pickup_datetime", "dropoff_datetime","PULocationID", "DOLocationID", "trip_miles")
    .count()
    .filter(col("count") > 1)
)

duplicates_df.count()

175557

In [17]:
duplicates_df.limit(2).show()

+-----------------+-------------------+------------+------------+-----+
|hvfhs_license_num|    pickup_datetime|PULocationID|DOLocationID|count|
+-----------------+-------------------+------------+------------+-----+
|           HV0003|2023-01-01 10:49:31|         261|         161|    2|
|           HV0003|2023-01-05 11:58:51|         138|         265|    2|
+-----------------+-------------------+------------+------------+-----+



In [20]:
df_dup_filtered = df.filter(
    (col("hvfhs_license_num") == "HV0003") &
    (col("pickup_datetime") == to_timestamp(lit("2023-01-01 10:49:31"))) & (col("PULocationID") == "261") & (col("DOLocationID") == "161")
)
df_dup_filtered.show()

+-----------------+--------------------+--------------------+-------------------+-------------------+-------------------+-------------------+------------+------------+----------+---------+-------------------+-----+----+---------+--------------------+-----------+----+----------+-------------------+-----------------+------------------+----------------+--------------+
|hvfhs_license_num|dispatching_base_num|originating_base_num|   request_datetime|  on_scene_datetime|    pickup_datetime|   dropoff_datetime|PULocationID|DOLocationID|trip_miles|trip_time|base_passenger_fare|tolls| bcf|sales_tax|congestion_surcharge|airport_fee|tips|driver_pay|shared_request_flag|shared_match_flag|access_a_ride_flag|wav_request_flag|wav_match_flag|
+-----------------+--------------------+--------------------+-------------------+-------------------+-------------------+-------------------+------------+------------+----------+---------+-------------------+-----+----+---------+--------------------+-----------+--

In [38]:
# invalid Trip duration
df = df.withColumn(
    "trip_seconds",
     unix_timestamp(col("dropoff_datetime")) - unix_timestamp(col("pickup_datetime"))
)
df.filter(col("trip_seconds") <= 0).count()

8507

In [39]:
df_seconds.filter(col("trip_seconds") <= 0).limit(5).show()

+-----------------+--------------------+--------------------+-------------------+-------------------+-------------------+-------------------+------------+------------+----------+---------+-------------------+-----+----+---------+--------------------+-----------+----+----------+-------------------+-----------------+------------------+----------------+--------------+--------------------+------------+
|hvfhs_license_num|dispatching_base_num|originating_base_num|   request_datetime|  on_scene_datetime|    pickup_datetime|   dropoff_datetime|PULocationID|DOLocationID|trip_miles|trip_time|base_passenger_fare|tolls| bcf|sales_tax|congestion_surcharge|airport_fee|tips|driver_pay|shared_request_flag|shared_match_flag|access_a_ride_flag|wav_request_flag|wav_match_flag|        trip_minutes|trip_seconds|
+-----------------+--------------------+--------------------+-------------------+-------------------+-------------------+-------------------+------------+------------+----------+---------+--------

In [ ]:
# removing the above invalid trips
df = df.filter(col("trip_seconds") > 0)

In [51]:
# ensuring that trip_time corresponds to (dropoff_datetime - pickup_datetime) within 1 second
df = df.withColumn(
    "seconds_diff",
     (col("trip_time") - col("trip_seconds"))
)
df = df.withColumn("absolute_diff", F.abs(df["seconds_diff"]))
df.filter((col("absolute_diff")) >1).count()

26029

In [53]:
df_seconds.filter((col("absolute_diff")) >1).limit(5).show()

+-----------------+--------------------+--------------------+-------------------+-------------------+-------------------+-------------------+------------+------------+----------+---------+-------------------+-----+-----+---------+--------------------+-----------+-----+----------+-------------------+-----------------+------------------+----------------+--------------+--------------------+------------+------------+-------------+
|hvfhs_license_num|dispatching_base_num|originating_base_num|   request_datetime|  on_scene_datetime|    pickup_datetime|   dropoff_datetime|PULocationID|DOLocationID|trip_miles|trip_time|base_passenger_fare|tolls|  bcf|sales_tax|congestion_surcharge|airport_fee| tips|driver_pay|shared_request_flag|shared_match_flag|access_a_ride_flag|wav_request_flag|wav_match_flag|        trip_minutes|trip_seconds|seconds_diff|absolute_diff|
+-----------------+--------------------+--------------------+-------------------+-------------------+-------------------+-----------------

In [ ]:
# removing more incorrect trip times
df = df.filter((col("absolute_diff")) <=1)

In [54]:
# outliers (Using Interquartile Range method)
def IQR_outlier_removal(df, column_name):
  quantiles = df.approxQuantile(column_name, [0.25, 0.75], 0.0)
  q1 = quantiles[0]
  q3 = quantiles[1]
  iqr = q3 - q1
  lower_val = q1 - 1.5 * iqr
  upper_val = q3 + 1.5 * iqr
  clean_df = df.filter((col(column_name) >= lower_val) & (col(column_name) <= upper_val))
  return clean_df

df_clean1 = IQR_outlier_removal(df, "trip_time")
df_clean2 = IQR_outlier_removal(df_clean1, "trip_miles")
df_clean2.count()

ERROR:root:KeyboardInterrupt while sending command.
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
  File "/usr/local/lib/python3.13/dist-packages/py4j/clientserver.py", line 535, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ~~~~~~~~~~~~~~~~~~~~^^
  File "/usr/lib/python3.13/socket.py", line 723, in readinto
    return self._sock.recv_into(b)
           ~~~~~~~~~~~~~~~~~~~~^^^
KeyboardInterrupt


KeyboardInterrupt: 

In [55]:
# bias
# checking which pick-up locations are not attended to
PULocation_counts = df_clean.groupBy("PULocationID").count()
PULocation_counts.orderBy(F.col("count").asc()).show(10)

# checking if a certain suburb gets charged more when there is no congestion surcharge

NameError: name 'df_clean' is not defined

In [ ]:
#checking which drop off zones do not get attended to
DOLocation_counts = df_clean.groupBy("DOLocationID").count()
DOLocation_counts.orderBy(F.col("count").asc()).show(10)

In [57]:
# check for operational time bias
df_clean = df.withColumn("Year", F.year("pickup_datetime")) \
              .withColumn("Month", F.month("pickup_datetime")) \
              .withColumn("Day", F.dayofmonth("pickup_datetime")) \
              .withColumn("weekday_name", F.date_format("pickup_datetime", "EEEE")) \
              .withColumn("period", F.date_format("pickup_datetime", "a"))


In [59]:
df_clean.limit(1).show()

ERROR:root:KeyboardInterrupt while sending command.
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
  File "/usr/local/lib/python3.13/dist-packages/py4j/clientserver.py", line 535, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ~~~~~~~~~~~~~~~~~~~~^^
  File "/usr/lib/python3.13/socket.py", line 723, in readinto
    return self._sock.recv_into(b)
           ~~~~~~~~~~~~~~~~~~~~^^^
KeyboardInterrupt


KeyboardInterrupt: 

In [ ]:
df.groupBy("period").count().orderBy("count", ascending=False).show()

In [ ]:
df.groupBy("weekday_name").count().orderBy("count", ascending=False).show()

**5. Exploratory Data Analysis**

In [ ]:
Year_df = df_clean.groupBy("Year").count().toPandas()


plt.figure(figsize=(8, 5))
sns.barplot(x="Year", y="Count", data=Year_df)
plt.title("Count Plot")
plt.xticks(rotation=45)
plt.show()

In [ ]:
Month_df = df_clean.groupBy("Month").count().toPandas()


plt.figure(figsize=(8, 5))
sns.barplot(x="Month", y="Count", data=Month_df)
plt.title("Count Plot")
plt.xticks(rotation=45)
plt.show()

In [ ]:
Day_df = df_clean.groupBy("weekday_name").count().toPandas()


plt.figure(figsize=(8, 5))
sns.barplot(x="Day of the week", y="Count", data=Day_df)
plt.title("Count Plot")
plt.xticks(rotation=45)
plt.show()

In [ ]:
shared_req_flag_df = df_clean.groupBy("shared_request_flag").count().toPandas()


plt.figure(figsize=(8, 5))
sns.barplot(x="Shared Request Flag", y="Count", data=shared_req_flag_df)
plt.title("Count Plot")
plt.xticks(rotation=45)
plt.show()

In [ ]:
shared_match_flag_df = df_clean.groupBy("shared_match_flag").count().toPandas()


plt.figure(figsize=(8, 5))
sns.barplot(x="Shared Match Flag", y="Count", data=shared_match_flag_df)
plt.title("Count Plot")
plt.xticks(rotation=45)
plt.show()

In [ ]:
access_a_ride_flag_df = df_clean.groupBy("access_a_ride_flag").count().toPandas()


plt.figure(figsize=(8, 5))
sns.barplot(x="Access a ride Flag", y="Count", data=access_a_ride_flag_df)
plt.title("Count Plot")
plt.xticks(rotation=45)
plt.show()

In [ ]:
wav_request_flag_df = df_clean.groupBy("wav_request_flag").count().toPandas()


plt.figure(figsize=(8, 5))
sns.barplot(x="Wav Request Flag", y="Count", data=wav_request_flag_df)
plt.title("Count Plot")
plt.xticks(rotation=45)
plt.show()

In [ ]:
wav_match_flag_df = df_clean.groupBy("wav_match_flag").count().toPandas()


plt.figure(figsize=(8, 5))
sns.barplot(x="Wav Match Flag", y="Count", data=wav_match_flag_df)
plt.title("Count Plot")
plt.xticks(rotation=45)
plt.show()

## 1.Setup

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install pyspark -q

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Big_data_project") \
    .getOrCreate()

spark

## 2. Load Files

In [ ]:
from pyspark.sql import functions as F

file_path = "drive/MyDrive/Big_data_project/Data"

df = spark.read.parquet(file_path)


print("Rows:", df.count())
print("Columns:", len(df.columns))
df.printSchema()

## 3. Dataset Characteristics

In [ ]:
df.describe().show()

In [ ]:
df.limit(5).toPandas()

In [ ]:
#Schema
df.printSchema()

In [ ]:
#Data date range
df.select(
    F.min("pickup_datetime"),
    F.max("pickup_datetime")
).show()

In [ ]:
#Dataset Size
import os

total_size = 0

for file in os.listdir("/content"):
    if file.endswith(".parquet"):
        total_size += os.path.getsize("/content/" + file)

print(f"Dataset size: {total_size/(1024**3):.2f} GB")

In [ ]:
#Records per year
df.groupBy(F.year("pickup_datetime").alias("Year")) \
  .count() \
  .orderBy("Year") \
  .show()

## 4. Data Quality Analysis

In [ ]:
#Missing Values
from pyspark.sql.functions import col, when, count

missing = df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df.columns
])

missing.show()

In [ ]:
#Duplicate Records
duplicates = df.count() - df.dropDuplicates().count()
print("Duplicate Records:", duplicates)

In [ ]:
#invalid Trip duration
df = df.withColumn(
    "trip_minutes",
    (F.col("dropoff_datetime").cast("long") -
     F.col("pickup_datetime").cast("long"))/60
)

df.filter(F.col("trip_minutes") < 0).count()

## 5. Exploratory Data Analysis (EDA)

In [ ]:
#Total trips
total_trips = df.count()
print(total_trips)

In [ ]:
#Trips by Month
monthly = df.groupBy(
    F.year("pickup_datetime").alias("Year"),
    F.month("pickup_datetime").alias("Month")
).count().orderBy("Year","Month")

monthly.show()

In [ ]:
#PLot
monthly_pd = monthly.toPandas()

import matplotlib.pyplot as plt

plt.figure(figsize=(12,5))
plt.plot(range(len(monthly_pd)), monthly_pd["count"])
plt.title("Trips Per Month")
plt.ylabel("Trips")
plt.show()

In [ ]:
#Trips per year

from pyspark.sql.functions import year

yearly = df.groupBy(
    year("pickup_datetime").alias("Year")
).count()

yearly.show()

In [ ]:
#PLot
yearly_pd = yearly.toPandas()

import matplotlib.pyplot as plt

plt.figure(figsize=(12,5))
plt.plot(range(len(yearly_pd)), yearly_pd["count"])
plt.title("Trips Per Month")
plt.ylabel("Trips")
plt.show()

In [ ]:
#Average Trip Distance
df.select(
    F.avg("trip_miles").alias("Average Distance")
).show()

In [ ]:
#Average Fare
df.select(
    F.avg("base_passenger_fare").alias("Average Fare")
).show()

In [ ]:
#Top Pickup Zone
df.groupBy("PULocationID") \
  .count() \
  .orderBy(F.desc("count")) \
  .show(10)

In [ ]:
#Top Drop off zones

df.groupBy("DOLocationID") \
  .count() \
  .orderBy("count", ascending=False) \
  .show(10)

In [ ]:
#Trips by Hour
hourly = df.groupBy(
    F.hour("pickup_datetime").alias("Hour")
).count().orderBy("Hour")

hourly.show()

In [ ]:
#Plot
hour_pd = hourly.toPandas()

plt.figure(figsize=(10,5))
plt.bar(hour_pd["Hour"], hour_pd["count"])
plt.title("Trips by Hour")
plt.show()